In [ ]:
# NSL-KDD Intrusion Detection System
# GCN + BiGRU Architecture with Two-Stage Pipeline
import os
import warnings
warnings.filterwarnings('ignore')
SEED = 42


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization,
                                      GRU, Bidirectional)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, accuracy_score
)
from sklearn.feature_selection import VarianceThreshold
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.preprocessing import LabelEncoder as LabelEncoderAttack

tf.random.set_seed(SEED)
np.random.seed(SEED)
print(f'TensorFlow: {tf.__version__}')


In [ ]:
# NSL-KDD column names (41 features + label + difficulty)
NSL_COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login',
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
    'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

# Attack-type to category mapping
DOS_SET   = {'back','land','neptune','pod','smurf','teardrop',
             'apache2','udpstorm','processtable','worm'}
PROBE_SET = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L_SET   = {'ftp_write','guess_passwd','imap','multihop','phf','spy',
             'warezclient','warezmaster','sendmail','named',
             'snmpattack','snmpguess','xlock','xsnoop','httptunnel'}
U2R_SET   = {'buffer_overflow','loadmodule','perl','rootkit',
             'sqlattack','xterm','ps'}

def map_attack_cat(label):
    l = label.lower().strip()
    if l == 'normal': return 'Normal'
    if l in DOS_SET:   return 'DoS'
    if l in PROBE_SET: return 'Probe'
    if l in R2L_SET:   return 'R2L'
    if l in U2R_SET:   return 'U2R'
    return 'DoS'  # unknown attacks treated as DoS

df_train = pd.read_csv('./dataset/nslkdd/KDDTrain+.txt',
                        header=None, names=NSL_COLUMNS)
df_test_raw = pd.read_csv('./dataset/nslkdd/KDDTest+.txt',
                           header=None, names=NSL_COLUMNS)

# Map raw attack label -> category
df_train['attack_cat'] = df_train['label'].apply(map_attack_cat)
df_test_raw['attack_cat'] = df_test_raw['label'].apply(map_attack_cat)

df = df_train.copy()
print(f'Training set : {df_train.shape}')
print(f'Test set     : {df_test_raw.shape}')
print(f'\nAttack categories (train):')
print(df['attack_cat'].value_counts().to_string())


In [ ]:
df.head()


In [ ]:
df.columns


In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
print('=== Data Quality ===')
print(f'Rows          : {len(df):,}')
print(f'Columns       : {len(df.columns)}')
print(f'NaN values    : {df.isnull().sum().sum():,}')
print(f'Inf values    : {np.isinf(numeric_df).sum().sum():,}')
print(f'Duplicates    : {df.duplicated().sum():,}')
print(f'\nCategorical cols: {list(df.select_dtypes(include="object").columns)}')


In [ ]:
df_test_raw = df_test_raw.drop_duplicates()
df = df.drop_duplicates()
print(f'After dedup — Train: {df.shape}  Test: {df_test_raw.shape}')


In [ ]:
LABEL_COL  = 'attack_cat'
BENIGN_STR = 'Normal'
print(f'Using label column : {LABEL_COL}')
print(f'Benign label value : {BENIGN_STR}')
print(f'\nClass distribution:')
print(df[LABEL_COL].value_counts().to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
fig.suptitle('NSL-KDD — Class Distribution', fontsize=13, fontweight='bold')

counts = df[LABEL_COL].value_counts()
counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black', width=0.7)
axes[0].set_title('Attack Categories')
axes[0].set_xlabel('Label'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis='y', alpha=0.3)

is_benign = df[LABEL_COL] == BENIGN_STR
binary_cnts = pd.Series({'Normal (0)': is_benign.sum(), 'Attack (1)': (~is_benign).sum()})
binary_cnts.plot(kind='bar', ax=axes[1],
                  color=['#4CAF50', '#F44336'], edgecolor='black', width=0.5)
axes[1].set_title('Binary: Normal vs Attack')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Binary label: 0 = Normal, 1 = Attack
df['binary_label'] = (df[LABEL_COL] != BENIGN_STR).astype(int)
y = df['binary_label'].values
print('Binary label distribution:')
print(f'  Normal (0) : {(y==0).sum():>10,}  ({(y==0).mean()*100:.1f}%)')
print(f'  Attack (1) : {(y==1).sum():>10,}  ({(y==1).mean()*100:.1f}%)')


In [ ]:
# One-hot encode categorical features: protocol_type, service, flag
CAT_COLS = ['protocol_type', 'service', 'flag']
DROP_COLS = ['label', 'attack_cat', 'binary_label', 'difficulty'] + CAT_COLS
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# Numeric features
X_num = df.drop(columns=DROP_COLS).select_dtypes(include=[np.number])

# One-hot encode categoricals (fit on train only to prevent leakage)
X_cat = pd.get_dummies(df[CAT_COLS], prefix=CAT_COLS, drop_first=False)

X = pd.concat([X_num, X_cat], axis=1)
print(f'Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features')
print(f'  Numeric : {X_num.shape[1]}  |  One-hot : {X_cat.shape[1]}')


In [ ]:
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
print(f'After cleanup — NaN: {X.isnull().sum().sum()} | Inf: {np.isinf(X.select_dtypes(include=np.number)).sum().sum()}')


In [ ]:
vt = VarianceThreshold(threshold=1e-4)
X_vt = vt.fit_transform(X)
SELECTED_FEATURES = X.columns[vt.get_support()].tolist()
X = pd.DataFrame(X_vt, columns=SELECTED_FEATURES)
NUM_FEATURES = X.shape[1]
print(f'Features kept : {NUM_FEATURES}  (removed {len(vt.get_support()) - NUM_FEATURES} low-variance)')


In [ ]:
# Prepare test set with same pipeline
DROP_TEST = ['label', 'attack_cat', 'binary_label', 'difficulty'] + CAT_COLS
DROP_TEST = [c for c in DROP_TEST if c in df_test_raw.columns]

df_test_raw['binary_label'] = (df_test_raw[LABEL_COL] != BENIGN_STR).astype(int)

X_test_num = df_test_raw.drop(columns=DROP_TEST).select_dtypes(include=[np.number])
X_test_cat = pd.get_dummies(df_test_raw[CAT_COLS], prefix=CAT_COLS, drop_first=False)
X_test_raw_df = pd.concat([X_test_num, X_test_cat], axis=1)
X_test_raw_df = X_test_raw_df.replace([np.inf, -np.inf], np.nan)

# Align test columns with training (fill missing one-hot cols with 0)
for col in SELECTED_FEATURES:
    if col not in X_test_raw_df.columns:
        X_test_raw_df[col] = 0
X_test_raw_df = X_test_raw_df[SELECTED_FEATURES]
X_test_raw_df = X_test_raw_df.fillna(X.median())

y_test      = df_test_raw['binary_label'].values
y_type_test = df_test_raw[LABEL_COL].astype(str).values

X_arr      = X.values
y_type_arr = df[LABEL_COL].astype(str).values

X_train, X_val, y_train, y_val, y_type_train, y_type_val = train_test_split(
    X_arr, y, y_type_arr, test_size=0.15, random_state=SEED, stratify=y
)
X_test = X_test_raw_df.values

print(f'Train : {X_train.shape}  Normal:{(y_train==0).sum():,}  Attack:{(y_train==1).sum():,}')
print(f'Val   : {X_val.shape}  Normal:{(y_val==0).sum():,}  Attack:{(y_val==1).sum():,}')
print(f'Test  : {X_test.shape}  Normal:{(y_test==0).sum():,}  Attack:{(y_test==1).sum():,}')


In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
print(f'Scaled — Train mean={X_train.mean():.4f} | std={X_train.std():.4f}')


In [ ]:
# Step 1: Attack-aware oversampling — cap multiplier to avoid extreme synthetic inflation
# U2R has only 52 samples; 177x SMOTE would create low-quality data
X_train_attacks = X_train[y_train == 1]
y_type_train_attacks = y_type_train[y_train == 1]
X_train_normal = X_train[y_train == 0]

attack_counts = Counter(y_type_train_attacks)
max_attack = max(attack_counts.values())
# Target: at most 8x current count OR 20% of largest class, whichever is smaller
smote_target = {
    k: min(int(max_attack * 0.20), max(v, v * 8))
    for k, v in attack_counts.items()
}

le_att = LabelEncoderAttack()
y_att_enc = le_att.fit_transform(y_type_train_attacks)
smote_att_strategy = {int(le_att.transform([k])[0]): smote_target[k]
                       for k in smote_target if smote_target[k] > attack_counts[k]}
if smote_att_strategy:
    smote_att = SMOTE(random_state=SEED, k_neighbors=3,
                      sampling_strategy=smote_att_strategy)
    X_att_res, y_att_enc_res = smote_att.fit_resample(X_train_attacks, y_att_enc)
else:
    X_att_res, y_att_enc_res = X_train_attacks, y_att_enc
y_att_res_types = le_att.inverse_transform(y_att_enc_res)

X_train_pre = np.vstack([X_train_normal, X_att_res])
y_train_pre = np.concatenate([np.zeros(len(X_train_normal)), np.ones(len(X_att_res))])
y_type_train_pre = np.concatenate([y_type_train[y_train == 0], y_att_res_types])

print("Attack category distribution after capped oversampling:")
for k, v in sorted(Counter(y_att_res_types).items()):
    orig = attack_counts[k]
    print(f"  {k:10s}: {orig:,} -> {v:,}  ({v/orig:.1f}x)")

# Step 2: Binary SMOTE
print("
Applying binary SMOTE...")
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train_pre, y_train_pre)
print(f"Before: Normal={(y_train_pre==0).sum():,}  Attack={(y_train_pre==1).sum():,}")
print(f"After : Normal={(y_train_bal==0).sum():,}  Attack={(y_train_bal==1).sum():,}")


In [ ]:
TIMESTEPS = 1

corr = np.corrcoef(X_train_bal, rowvar=False)
corr = np.nan_to_num(np.abs(corr), nan=0.0, posinf=0.0, neginf=0.0)
np.fill_diagonal(corr, 1.0)

TOP_K = 8
idx = np.argsort(corr, axis=1)[:, -TOP_K:]
mask = np.zeros_like(corr)
rows = np.arange(corr.shape[0])[:, None]
mask[rows, idx] = 1.0
A = corr * mask
A = np.maximum(A, A.T)
np.fill_diagonal(A, 1.0)

deg = A.sum(axis=1)
deg_inv_sqrt = np.power(deg + 1e-8, -0.5)
A_norm = (deg_inv_sqrt[:, None] * A) * deg_inv_sqrt[None, :]

X_train_lstm = X_train_bal.reshape(-1, TIMESTEPS, NUM_FEATURES)
X_val_lstm   = X_val.reshape(-1, TIMESTEPS, NUM_FEATURES)
X_test_lstm  = X_test.reshape(-1, TIMESTEPS, NUM_FEATURES)

print('GCN + BiGRU input shapes:')
print(f'  Train : {X_train_lstm.shape}')
print(f'  Val   : {X_val_lstm.shape}')
print(f'  Test  : {X_test_lstm.shape}')
print(f'  Graph edges retained: {(A > 0).sum():,}')


In [ ]:
class SimpleGCN(tf.keras.layers.Layer):
    def __init__(self, units, adjacency, activation='relu', l2_reg=1e-3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.adjacency = tf.constant(adjacency, dtype=tf.float32)
        self.activation = tf.keras.activations.get(activation)
        self.l2_reg = l2_reg

    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.kernel = self.add_weight(
            name='kernel', shape=(in_dim, self.units),
            initializer='glorot_uniform', regularizer=l2(self.l2_reg), trainable=True)
        self.bias = self.add_weight(
            name='bias', shape=(self.units,),
            initializer='zeros', trainable=True)

    def call(self, inputs):
        x = tf.squeeze(inputs, axis=1)
        x = tf.matmul(x, self.adjacency)
        x = tf.matmul(x, self.kernel) + self.bias
        if self.activation is not None:
            x = self.activation(x)
        return tf.expand_dims(x, axis=1)


def build_gcn_bigru(num_features, adjacency, timesteps=1, dropout=0.3, reg=1e-3):
    model = Sequential(name='GCN_BiGRU_NSL_KDD')
    model.add(Input(shape=(timesteps, num_features)))
    model.add(SimpleGCN(num_features, adjacency=adjacency,
                        activation='relu', l2_reg=reg, name='gcn_front'))
    model.add(Bidirectional(GRU(128, return_sequences=True,
                                kernel_regularizer=l2(reg),
                                recurrent_regularizer=l2(reg))))
    model.add(BatchNormalization())
    model.add(Dropout(dropout))
    model.add(Bidirectional(GRU(64, return_sequences=False,
                                kernel_regularizer=l2(reg))))
    model.add(BatchNormalization())
    model.add(Dropout(dropout))
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(reg)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout / 2))
    model.add(Dense(32, activation='relu'))
    model.add(Dropout(dropout / 2))
    model.add(Dense(1, activation='sigmoid', name='anomaly_output'))
    return model


model = build_gcn_bigru(NUM_FEATURES, A_norm, TIMESTEPS)
model.summary()


In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    """Focal loss: focuses training on hard-to-classify samples (U2R, R2L)."""
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1 - p_t, gamma) * bce)
    return loss_fn

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)
print('Model compiled with Focal Loss (gamma=2.0, alpha=0.25)')


In [ ]:
cw = compute_class_weight('balanced', classes=np.unique(y_train_bal), y=y_train_bal)
CLASS_WEIGHTS = dict(enumerate(cw))
print(f'Class weights: {CLASS_WEIGHTS}')


In [ ]:
# Phase 1 — Initial Training (LR = 1e-3)
print('PHASE 1 — Initial Training')
print('='*50)

cb1 = [
    EarlyStopping(monitor='val_auc', patience=10,
                  restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5,
                      min_lr=1e-6, verbose=1),
    ModelCheckpoint('./dataset/nslkdd/best_phase1.weights.h5',
                    monitor='val_auc', save_best_only=True, mode='max',
                    save_weights_only=True, verbose=1)
]

h1 = model.fit(
    X_train_lstm, y_train_bal,
    validation_data=(X_val_lstm, y_val),
    epochs=50,
    batch_size=512,
    class_weight=CLASS_WEIGHTS,
    callbacks=cb1,
    verbose=1
)


In [ ]:
# Phase 2 — Fine-Tuning (LR = 1e-4)
print('PHASE 2 — Fine-Tuning')
print('='*50)

model = build_gcn_bigru(NUM_FEATURES, A_norm, TIMESTEPS)
model.load_weights('./dataset/nslkdd/best_phase1.weights.h5')
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

cb2 = [
    EarlyStopping(monitor='val_auc', patience=8,
                  restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4,
                      min_lr=1e-7, verbose=1),
    ModelCheckpoint('./dataset/nslkdd/best_finetuned.weights.h5',
                    monitor='val_auc', save_best_only=True, mode='max',
                    save_weights_only=True, verbose=1)
]

h2 = model.fit(
    X_train_lstm, y_train_bal,
    validation_data=(X_val_lstm, y_val),
    epochs=30,
    batch_size=256,
    class_weight=CLASS_WEIGHTS,
    callbacks=cb2,
    verbose=1
)


In [ ]:
best_model = build_gcn_bigru(NUM_FEATURES, A_norm, TIMESTEPS)
best_model.load_weights('./dataset/nslkdd/best_finetuned.weights.h5')

y_prob = best_model.predict(X_test_lstm, batch_size=512, verbose=0).flatten()

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
opt_idx = np.argmax(tpr - fpr)
OPT_THRESHOLD = float(thresholds[opt_idx])
y_pred = (y_prob >= OPT_THRESHOLD).astype(int)

print(f'Optimal threshold (Youden J): {OPT_THRESHOLD:.4f}')


In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
f1  = f1_score(y_test, y_pred)

print('=' * 50)
print('   GCN + BiGRU NSL-KDD IDS  —  TEST RESULTS')
print('=' * 50)
print(f'  Accuracy  :  {acc * 100:.2f}%')
print(f'  AUC-ROC   :  {auc * 100:.2f}%')
print(f'  F1-Score  :  {f1  * 100:.2f}%')
print()
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GCN + BiGRU NSL-KDD IDS — Test Evaluation', fontsize=13, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, None] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'],
            ax=axes[0], linewidths=0.5, annot_kws={'size': 13})
axes[0].set_title('Confusion Matrix (%)', fontweight='bold')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted Label')
for i in range(2):
    for j in range(2):
        axes[0].text(j + 0.5, i + 0.72, f'n={cm[i, j]:,}',
                     ha='center', va='center', fontsize=9, color='dimgray')

# ROC Curve
axes[1].plot(fpr, tpr, color='#1565C0', lw=2.5, label=f'GCN+BiGRU  AUC={auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'gray', lw=1, ls='--', label='Random baseline')
axes[1].scatter(fpr[opt_idx], tpr[opt_idx], s=120, color='red', zorder=5,
                label=f'Best threshold={OPT_THRESHOLD:.3f}')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('./evaluation_nslkdd.png', dpi=150)
plt.show()


In [ ]:
# Attack-wise confusion matrix: true label vs predicted label (%)
attack_mask = y_type_test != BENIGN_STR
attack_true = y_type_test[attack_mask]
attack_pred_binary = np.where(y_pred[attack_mask] == 1, attack_true, BENIGN_STR)

all_attack_types = sorted(set(attack_true))
attack_cm = pd.crosstab(
    pd.Series(attack_true, name='True Attack Type'),
    pd.Series(attack_pred_binary, name='Predicted Label')
)
for col in all_attack_types + [BENIGN_STR]:
    if col not in attack_cm.columns:
        attack_cm[col] = 0
col_order = sorted([c for c in attack_cm.columns if c != BENIGN_STR]) + [BENIGN_STR]
attack_cm = attack_cm[col_order]
attack_cm_pct = attack_cm.div(attack_cm.sum(axis=1), axis=0).fillna(0.0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(attack_cm_pct, annot=True, fmt='.1f', cmap='YlGnBu',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% of true samples'},
            vmin=0, vmax=100)
ax.set_title('Attack-Type Confusion Matrix (%) — Stage-1 Binary', fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Attack Type')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print('\nDetection recall by attack category:')
for att in all_attack_types:
    row = attack_cm.loc[att]
    detected = row.get(att, 0)
    total = row.sum()
    print(f'  {att:10s}: {detected/total*100:.1f}%  (n={int(total):,})')


In [ ]:
# =================================================================
# STAGE 2 — Multi-class Attack-Category Classifier (GBM)
# Improves U2R and R2L detection (rarest categories)
# NOTE: X_train_pre is already scaled (built from scaled X_train)
# =================================================================
from sklearn.ensemble import GradientBoostingClassifier

# X_train_pre is already scaled — do NOT call scaler.transform again
X_s2_raw   = X_train_pre[y_train_pre == 1]
y_s2_types = y_type_train_pre[y_train_pre == 1]

le_s2 = LabelEncoderAttack()
y_s2_enc = le_s2.fit_transform(y_s2_types)
smote_s2 = SMOTE(random_state=SEED, k_neighbors=3)
X_s2_bal, y_s2_bal_enc = smote_s2.fit_resample(X_s2_raw, y_s2_enc)
y_s2_bal = le_s2.inverse_transform(y_s2_bal_enc)

print("Stage-2 training distribution (after SMOTE):")
for k, v in sorted(Counter(y_s2_bal).items()):
    print(f"  {k:10s}: {v:,}")

stage2_clf = GradientBoostingClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=SEED
)
stage2_clf.fit(X_s2_bal, y_s2_bal)
print("Stage-2 GBM classifier trained.")


In [ ]:
# Combined pipeline: Stage-1 flags attacks, Stage-2 identifies category.
# U2R and R2L get a lower binary threshold to reduce false negatives.

HARD_TYPES = {'U2R', 'R2L'}
LOWER_THRESHOLD = OPT_THRESHOLD * 0.65

attack_mask = y_type_test != BENIGN_STR
stage2_pred_types = stage2_clf.predict(X_test[attack_mask])

y_pred_combined = y_pred.copy()
attack_indices = np.where(attack_mask)[0]
for local_i, global_i in enumerate(attack_indices):
    if stage2_pred_types[local_i] in HARD_TYPES:
        y_pred_combined[global_i] = int(y_prob[global_i] >= LOWER_THRESHOLD)

attack_true_c = y_type_test[attack_mask]
attack_pred_combined_labels = np.where(
    y_pred_combined[attack_mask] == 1,
    stage2_pred_types,
    BENIGN_STR
)

all_attack_types_c = sorted(set(attack_true_c))
attack_cm_combined = pd.crosstab(
    pd.Series(attack_true_c, name='True Attack Type'),
    pd.Series(attack_pred_combined_labels, name='Predicted Label')
)
for col in all_attack_types_c + [BENIGN_STR]:
    if col not in attack_cm_combined.columns:
        attack_cm_combined[col] = 0
col_order_c = sorted([c for c in attack_cm_combined.columns if c != BENIGN_STR]) + [BENIGN_STR]
attack_cm_combined = attack_cm_combined[col_order_c]

print('Attack detection recall — Original vs Combined:')
print(f'{"Category":10s}  {"Original %":>12}  {"Combined %":>12}  {"Improvement":>12}')
print('-' * 54)
for att in all_attack_types_c:
    orig_r = attack_cm.loc[att, att] / attack_cm.loc[att].sum() * 100 if att in attack_cm.columns else 0.0
    comb_r = attack_cm_combined.loc[att, att] / attack_cm_combined.loc[att].sum() * 100 if att in attack_cm_combined.columns else 0.0
    print(f'  {att:10s}  {orig_r:>11.1f}%  {comb_r:>11.1f}%  {comb_r - orig_r:>+11.1f}%')

acc_combined = accuracy_score(y_test, y_pred_combined)
print(f'\nOverall accuracy — Original: {acc*100:.2f}%  |  Combined: {acc_combined*100:.2f}%')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('Attack-Type Confusion Matrix (%): Original vs Combined Pipeline',
             fontsize=13, fontweight='bold')

for ax, cm_data, title in zip(
    axes,
    [attack_cm, attack_cm_combined],
    ['Stage-1 Binary', 'Stage-1 + Stage-2 Combined']
):
    pct = cm_data.div(cm_data.sum(axis=1), axis=0).fillna(0) * 100
    sns.heatmap(pct, annot=True, fmt='.1f', cmap='YlGnBu',
                linewidths=0.5, ax=ax,
                cbar_kws={'label': '% of true samples'},
                vmin=0, vmax=100)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Attack Type')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('./evaluation_nslkdd_stage2.png', dpi=150)
plt.show()
